In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, message=".*escape sequence.*")

from textwrap import dedent
import httpx
import json
import requests
import pandas as pd
import numpy as np
import re
import requests
import uuid
from openai import OpenAI
import time
import os
from dotenv import load_dotenv, find_dotenv

from pydantic import BaseModel, Field
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser, BaseOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


load_dotenv(find_dotenv(usecwd=True))

pubmed_api_key= '6892c4129cef143ff92d11533848d2e0d908'#os.getenv("PUBMED_API_KEY", '')
use = 'olm'

if use == 'hf':
    os.environ["BASE_URL"] = "https://router.huggingface.co/v1"
    os.environ["MODEL_NAME"] ='Qwen/Qwen3-32B'
    os.environ["OPENAI_API_KEY"] = os.getenv("HUGGINGFACE_HUB_TOKEN2")
    
elif use == 'olm':
    os.environ["BASE_URL"] = "http://localhost:11434/v1"
    os.environ["MODEL_NAME"] ='qwen3:14b'
    os.environ["OPENAI_API_KEY"] = 'hey'
    
else:
    os.environ["BASE_URL"] = "https://gigachat.devices.sberbank.ru/api/v1"
    os.environ["MODEL_NAME"] ='GigaChat-Max'
    
    url = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"
    # Создадим идентификатор UUID (36 знаков)
    rq_uid = str(uuid.uuid4())
    payload={
      'scope': 'GIGACHAT_API_PERS'
    }
    headers = {
      'Content-Type': 'application/x-www-form-urlencoded',
      'Accept': 'application/json',
      'RqUID': rq_uid,
      'Authorization': f'Basic {os.getenv("GIGACHAT_AUTH")}'
    }

    response = requests.request("POST", url, headers=headers, data=payload, verify=False)
    os.environ["OPENAI_API_KEY"] = response.json()['access_token']
    

os.environ['TEMPERATURE'] = '0'


# for importing trialmind, api_key should be set beforehand
from trialmind.pubmed import pmid2papers, PubmedAPIWrapper, pmid2biocxml, parse_bioc_xml, pmid2fulltext
from trialmind.api import StudyCharacteristicsExtraction, ScreeningCriteriaGeneration,\
                            LiteratureScreening, ScreeningCriteriaCTGeneration,\
                            CTScreening
from trialmind.retrievers import split_text_into_chunks
import extract
import time

%load_ext autoreload
%autoreload 2

In [50]:
df.columns[1::2] + ' clm2'

Index(['Adrenocortical cancer clm2', 'Basal cell carcinoma clm2',
       'Breast cancer clm2', 'Carcinoid tumour/Neuroendocrine tumours clm2',
       'Cervical cancer clm2', 'Cholangiocarcinoma clm2',
       'Colorectal cancer clm2', 'Endometrial cancer clm2',
       'Esophageal cancer clm2', 'Fallopian tube cancer clm2', 'GIST clm2',
       'Gastric cancer clm2', 'Germ cell tumour clm2', 'Glioblastoma clm2',
       'Glioma clm2', 'Head and neck cancer clm2',
       'Hepatocellular carcinoma clm2', 'Lung cancer (NSCLC) clm2',
       'Lung cancer (SCLC) clm2',
       'Lymphoma, leukemia (hematological malign) clm2', 'Melanoma clm2',
       'Meningioma clm2', 'Mesothelioma clm2', 'Others clm2',
       'Ovarian cancer clm2', 'Pancreatic cancer clm2',
       'Pheochromocytoma/Paraganglioma clm2', 'Prostate cancer clm2',
       'Renal cancer clm2', 'Sarcoma clm2', 'Schwannoma clm2',
       'Thyroid cancer clm2', 'Urothelial cancer clm2'],
      dtype='str')

In [4]:
'''
http://www.accessdata.fda.gov/drugsatfda_docs/label/2014/020449s073lbl.pdf for Docetaxel on:
https://www.accessdata.fda.gov/scripts/cder/daf/index.cfm?event=BasicSearch.process
11/14/2014 	SUPPL-73 	Labeling-Package Insert 	Label (PDF)
'''
df = pd.read_excel('docs/clinical_trials_2send.xlsx')
df.iloc[49:50].dropna(axis=1)

,Drug,Breast cancer,Esophageal cancer,Fallopian tube cancer,Unnamed: 20,Gastric cancer,Head and neck cancer,Lung cancer (NSCLC),Ovarian cancer,Unnamed: 50,Pancreatic cancer,Prostate cancer,Sarcoma,Unnamed: 60,Urothelial cancer,Unnamed: 66
49,Docetaxel,"Marketed, http://www.accessdata.fda.gov/drugsa...",Marketed (adenoca of the gastroesophageal junc...,Phase III completed,https://clinicaltrials.gov/ct2/show/NCT0000312...,Marketed http://www.accessdata.fda.gov/drugsat...,Marketed http://www.accessdata.fda.gov/drugsat...,Marketed http://www.accessdata.fda.gov/drugsat...,Phase III completed,https://clinicaltrials.gov/ct2/show/NCT0000312...,Discontinued,Marketed http://www.accessdata.fda.gov/drugsat...,Phase III completed,https://clinicaltrials.gov/ct2/show/NCT0014257...,Phase III,https://clinicaltrials.gov/ct2/show/NCT0242612...


In [117]:
def extract_time(json):
    try:
        # Also convert to int since update_time will be string.  When comparing
        # strings, "10" is smaller than "2".
        return int(json['date'])
    except KeyError:
        return 0

def get_latest_label(res):
    # just taking the latest available label across ALL products with given active ingredient
    # OR?? should we browse by a-z drug index instead of 'search'?
    # https://www.accessdata.fda.gov/scripts/cder/daf/index.cfm?event=faq.page#searches_about
    all_t = []
    for treat in res.json()['results']:
        subms = []
        for i in treat['submissions']:
            for j in i.get('application_docs',[]):
                if j.get('type','')=='Label':
                    subms.append(j)
        if subms:
            # lines.sort() is more efficient than lines = lines.sorted()
            subms.sort(key=extract_time, reverse=True)
            all_t.append(subms[0])
        else:
            all_t.append({})
    all_t.sort(key=extract_time, reverse=True)
    return all_t[0]

In [84]:
res=requests.get('https://api.fda.gov/drug/drugsfda.json?limit=99&search=openfda.substance_name:%22docetaxel%22',
                )
res

<Response [200]>

In [118]:
get_latest_label(res)

{'id': '79979',
 'url': 'http://www.accessdata.fda.gov/drugsatfda_docs/label/2024/218711s000lbl.pdf',
 'date': '20241025',
 'type': 'Label'}

In [120]:
res2=requests.get('https://api.fda.gov/drug/drugsfda.json?limit=99&search=openfda.substance_name:%22Bevacizumab%22',
                )
get_latest_label(res2)

{'id': '82014',
 'url': 'http://www.accessdata.fda.gov/drugsatfda_docs/label/2025/761175s000lbl.pdf',
 'date': '20250425',
 'type': 'Label'}

In [115]:
pd.json_normalize(res.json()['results'][0])#.application_docs.values

,submissions,application_number,sponsor_name,products,openfda.application_number,openfda.brand_name,openfda.generic_name,openfda.manufacturer_name,openfda.product_ndc,openfda.product_type,openfda.route,openfda.substance_name,openfda.rxcui,openfda.spl_id,openfda.spl_set_id,openfda.package_ndc,openfda.unii
0,"[{'submission_type': 'ORIG', 'submission_numbe...",ANDA213768,GUANGDONG SUNHO,"[{'product_number': '001', 'reference_drug': '...",[ANDA213768],[DOCETAXEL],[DOCETAXEL],[Meitheal Pharmaceuticals Inc.],"[71288-133, 71288-152]",[HUMAN PRESCRIPTION DRUG],[INTRAVENOUS],[DOCETAXEL],"[1860480, 1860485]",[bf235693-ce1a-4f89-b33a-8aaf93efb275],[f970e352-3d43-4cc4-a859-503ec37b6e22],"[71288-133-01, 71288-152-04]",[15H5577CQD]
